In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("="*60)
print("STEP 1: PARAMETERS RECEIVED")
print("="*60)
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  table_name        : {table_name}")
print(f"  merge_keys        : {merge_keys}")
print(f"  load_type         : {load_type}")
print("="*60)

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
filter_expr = f"landing_timestamp = '{landing_timestamp}'"

print("="*60)
print("STEP 2: READING BRONZE TABLE")
print("="*60)
print(f"  Source table  : {source_table}")
print(f"  Filter        : {filter_expr}")
print(f"  Query equiv   : SELECT * FROM {source_table} WHERE {filter_expr}")
print("-"*60)

bronze_df = spark.read.table(source_table).filter(filter_expr)

row_count = bronze_df.count()
columns = bronze_df.columns
print(f"  Rows read     : {row_count}")
print(f"  Columns ({len(columns)}): {columns}")
print(f"  Schema        :")
bronze_df.printSchema()
print("="*60)


In [0]:
from pyspark.sql.functions import lit, current_timestamp

print("="*60)
print("STEP 3: ADDING AUDIT COLUMNS")
print("="*60)
print("  Adding column : insert_timestamp = current_timestamp()")
print("  Adding column : update_timestamp = current_timestamp()")
print("-"*60)

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print(f"  Final columns ({len(updated_df.columns)}): {updated_df.columns}")
print(f"  New schema    :")
updated_df.printSchema()
print("="*60)

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"

print("="*60)
print("STEP 4: WRITING TO SILVER TABLE")
print("="*60)
print(f"  Target table  : {target_table}")
print(f"  Load type     : {load_type.upper()}")
print("-"*60)

if load_type.upper() == "FULL":
    print("  Mode          : OVERWRITE (full load)")
    print(f"  Query equiv   : INSERT OVERWRITE {target_table} SELECT * FROM source_df")
    print("  Executing write...")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed successfully.")

elif load_type.upper() == "APPEND":
    print("  Mode          : APPEND")
    print(f"  Query equiv   : INSERT INTO {target_table} SELECT * FROM source_df")
    print("  Executing write...")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed successfully.")

elif load_type.upper() == "MERGE":
    print("  Mode          : MERGE (upsert)")
    table_exists = spark.catalog.tableExists(target_table)
    print(f"  Target exists : {table_exists}")

    if table_exists:
        keys = [k.strip() for k in merge_keys.split(",")]
        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

        # On update: refresh only update_timestamp, preserve insert_timestamp
        update_set = {
            c: f"source.{c}"
            for c in updated_df.columns
            if c not in keys and c != "insert_timestamp"
        }

        print(f"  Merge keys    : {keys}")
        print(f"  Merge cond    : {merge_condition}")
        print(f"  Update set    : {update_set}")
        print(f"  Query equiv   :")
        print(f"    MERGE INTO {target_table} AS target")
        print(f"    USING source_df AS source")
        print(f"    ON {merge_condition}")
        print(f"    WHEN MATCHED THEN UPDATE SET {', '.join([f'{k} = {v}' for k, v in update_set.items()])}")
        print(f"    WHEN NOT MATCHED THEN INSERT *")
        print("  Executing merge...")

        DeltaTable.forName(spark, target_table).alias("target").merge(
            updated_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(set=update_set) \
         .whenNotMatchedInsertAll() \
         .execute()
        print("  MERGE completed successfully.")
    else:
        print("  Target table does not exist — performing initial FULL load instead.")
        print(f"  Query equiv   : INSERT OVERWRITE {target_table} SELECT * FROM source_df")
        print("  Executing write...")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial full load completed successfully.")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

print("="*60)

In [0]:
final_count = updated_df.count()

print("="*60)
print("STEP 5: NOTEBOOK EXIT")
print("="*60)
print(f"  Rows processed : {final_count}")
print(f"  Exiting with   : {final_count}")
print("="*60)

dbutils.notebook.exit(str(final_count))